# Weather codes and sales per product group

**Purpose.** For every weather code and every product group, test whether turnover on days with that code differs from turnover on calm days. This is an exploration, not a model: it shows which kinds of weather go together with higher or lower sales.

> **Summary**
> - **Following this procedure, the weather code cannot be used for predicting sales.** Once code days and calm reference days (00–03) are matched by month, only 1 of 47 tests (code group × product group) is significant (p < 0.05): fog (4X) for croissants. That is fewer than chance alone would produce (about 2), and it does not pass the Bonferroni correction. Rain (6X) shows no significant effect for any product group.
> - **This does not necessarily mean that weather is irrelevant.** It may be an operationalization problem: one daily code summarises many observations in an undocumented way, it is missing on many days (not at random), and rare codes leave only a few days per test. Other measures of weather (temperature, cloud cover, or deviations from the usual weather for the month) may still capture an effect.

**Comparison.** The reference is the days with `Wettercode` **00–03** (no precipitation, only change of cloud cover). For each product group and each other code, the days with that code are compared with reference days **of the same product group and the same months**.

**Why month matching?** Weather and season are confounded. So both groups are **thinned to the same months**: in each month *m*, take as many days as the smaller side has (days with the code or reference days in month *m*, any year) and draw that number at random from **both** sides. Example: if January has 25 snow days and 6 reference days, 6 random snow days are compared with the 6 reference days. Both groups then have exactly the same number of days in every month, so season cannot explain a difference between them. The cost: fewer days per test. The random draw uses a fixed seed, so the result is the same on every run.

WMO codes for reference: 00–03 no precipitation (reference), 04–09 haze, 10–19 mist, lightning, precipitation in sight, 20–29 precipitation or fog in the past hour, 40–49 fog, 50–59 drizzle, 60–69 rain, 70–79 snow, 80–99 showers and thunderstorms.

**Statistics per combination**
- **change (€)** – mean turnover on days with the code minus mean turnover on reference days. Negative = less sold.
- **Cohen's d** – the same difference in standard deviations (pooled). Rough guide: |d| ≈ 0.2 small, 0.5 medium, 0.8 large.
- **p-value** – Welch's t-test (does not assume equal spread). p < 0.05 = significant.
- **days (each side)** – matched days per group after thinning; **days with code (all)** – all days with that code before thinning; **used (%)** – share of the code's days that could be matched.
- **summer days (%)** – share of the matched days in May–September (identical for both groups by construction).

**Only significant combinations (p < 0.05) are shown**, ordered by Cohen's d from highest to lowest.

The analysis is done for **code groups of ten** (0X, 1X, 2X, …).

## Data preparation

- Sales joined with weather on `Datum` (left join, sales as the base).
- Days without a weather code are left out: they have no code to test.
- 31 December is left out: New Year's Eve, not weather, drives the cake peaks.
- Combinations with fewer than 2 matched days per side cannot be tested and are skipped.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

BASE_URL = "https://raw.githubusercontent.com/opencampus-sh/intro-to-data-science-and-ml/main/data/bakery/"
GROUP_NAMES = {1: "Bread", 2: "Rolls", 3: "Croissant", 4: "Confectionery", 5: "Cake", 6: "Seasonal bread"}
REFERENCE_CODES = [0, 1, 2, 3]

sales = pd.read_csv(BASE_URL + "umsatzdaten_gekuerzt.csv", parse_dates=["Datum"])
weather = pd.read_csv(BASE_URL + "wetter.csv", parse_dates=["Datum"])

df = sales.merge(weather[["Datum", "Wettercode"]], on="Datum", how="left")
df = df[~((df["Datum"].dt.month == 12) & (df["Datum"].dt.day == 31))]
df = df[df["Wettercode"].notna()].copy()
df["Wettercode"] = df["Wettercode"].astype(int)

print("sales rows:", len(sales), "| rows with a weather code (31 Dec left out):", len(df))
print("reference rows (codes 00-03) per group:")
print(df[df["Wettercode"].isin(REFERENCE_CODES)].groupby("Warengruppe").size().rename(GROUP_NAMES).to_string())

sales rows: 9334 | rows with a weather code (31 Dec left out): 6990
reference rows (codes 00-03) per group:
Warengruppe
Bread             161
Rolls             161
Croissant         161
Confectionery     158
Cake              161
Seasonal bread     18


## Tests

For each product group and each code group: Cohen's d (pooled standard deviation) and Welch's t-test between the month-matched code days and reference days of the same group.

**Multiple testing.** Many tests are run here. At the 5 % level, about 1 in 20 tests is "significant" by chance alone. The column **Bonferroni** shows whether a result also passes the strict threshold 0.05 / (number of tests); results without it should be read with care.

In [4]:
def summer_share(rows):
    """Share of days in May-September, in %: shows whether a code is mostly summer or winter weather."""
    return round(rows["Datum"].dt.month.between(5, 9).mean() * 100)

def compare(code_rows, ref_rows):
    code_days, ref_days = code_rows["Umsatz"], ref_rows["Umsatz"]
    n1, n2 = len(code_days), len(ref_days)
    pooled_sd = (((n1 - 1) * code_days.var() + (n2 - 1) * ref_days.var()) / (n1 + n2 - 2)) ** 0.5
    return {
        "days (each side)": n1,
        "summer days (%)": summer_share(code_rows),
        "mean (€)": code_days.mean(),
        "reference mean (€)": ref_days.mean(),
        "change (€)": code_days.mean() - ref_days.mean(),
        "Cohen's d": (code_days.mean() - ref_days.mean()) / pooled_sd,
        "p-value": stats.ttest_ind(code_days, ref_days, equal_var=False).pvalue,
    }

def month_matched(test_rows, ref_rows, rng):
    """Thin both sides to the same months: in each month, draw n = min(code days, reference days)
    days at random from both sides (without replacement). Returns (code days, reference days)."""
    test_parts, ref_parts = [], []
    for month, t in test_rows.groupby(test_rows["Datum"].dt.month):
        pool = ref_rows[ref_rows["Datum"].dt.month == month]
        n = min(len(t), len(pool))
        test_parts.append(t.sample(n=n, random_state=rng))
        ref_parts.append(pool.sample(n=n, random_state=rng))
    return pd.concat(test_parts), pd.concat(ref_parts)

def run_tests(by):
    """Test every value of column `by` against month-matched reference days, per product group.
    Prints the test count and returns only significant rows (p < 0.05), ordered by Cohen's d."""
    rng = np.random.default_rng(42)  # fixed seed: same random draw on every run
    rows = []
    for group, g in df.groupby("Warengruppe"):
        ref = g[g["Wettercode"].isin(REFERENCE_CODES)]
        for value, c in g[~g["Wettercode"].isin(REFERENCE_CODES)].groupby(by):
            matched_code, matched_ref = month_matched(c, ref, rng)
            if len(matched_code) < 2:
                continue
            rows.append({"product group": f"{group} {GROUP_NAMES[group]}", by: value,
                         "days with code (all)": len(c),
                         "used (%)": round(len(matched_code) / len(c) * 100),
                         **compare(matched_code, matched_ref)})
    results = pd.DataFrame(rows)
    n_tests = len(results)
    alpha_bonferroni = 0.05 / n_tests
    results["Bonferroni"] = np.where(results["p-value"] < alpha_bonferroni, "yes", "no")
    print(f"tests run: {n_tests} | significant at 0.05: {(results['p-value'] < 0.05).sum()} "
          f"| expected by chance: about {n_tests * 0.05:.0f} | Bonferroni threshold: {alpha_bonferroni:.5f}")
    print(f"code days used after month matching: median {results['used (%)'].median():.0f} % "
          f"(range {results['used (%)'].min()}-{results['used (%)'].max()} %)")
    significant = (results[results["p-value"] < 0.05]
                   .sort_values("Cohen's d", ascending=False).reset_index(drop=True))
    significant["p-value"] = significant["p-value"].map(lambda p: "< 0.0001" if p < 0.0001 else f"{p:.4f}")
    return significant.round({"mean (€)": 1, "reference mean (€)": 1, "change (€)": 1, "Cohen's d": 2})

## Code groups (0X, 1X, 2X, …)

The codes are pooled into blocks of ten, which follow the WMO structure. Pooling gives more days per test than single codes would, so results are less fragile. The reference stays codes 00–03, so the block **0X contains only codes 04–09** (haze, smoke, dust).

| Block | Meaning |
|---|---|
| 0X | 04–09 haze, smoke, dust |
| 1X | mist, lightning, precipitation in sight |
| 2X | precipitation, fog or thunderstorm in the past hour |
| 4X | fog |
| 5X | drizzle |
| 6X | rain |
| 7X | snow, ice pellets |
| 8X | showers |
| 9X | thunderstorm |

(No 3X codes occur in the data.) Significant combinations (p < 0.05), ordered by Cohen's d.

In [3]:
df["code group"] = df["Wettercode"].map(lambda c: f"{c // 10}X")
run_tests("code group")

tests run: 47 | significant at 0.05: 1 | expected by chance: about 2 | Bonferroni threshold: 0.00106
code days used after month matching: median 88 % (range 15-100 %)


,product group,code group,days with code (all),used (%),days (each side),summer days (%),mean (€),reference mean (€),change (€),Cohen's d,p-value,Bonferroni
0,3 Croissant,4X,27,89,24,0,103.7,131.2,-27.5,-0.65,0.0290,no
